# MOSAIC Date / Datetime Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 – Overview | This document | — |
| 2 – Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 – Config | Parameters / widgets | — |
| 4 – Constants | Tags, sentinel dates, severity weights, standardization rules | All |
| 5 – Discovery | Table & column inventory from information_schema | — |
| 6 – Profiler | Stats for all date/datetime candidates; timezone companion scan | §Format, §TZ, §Validation |
| 7 – AI Classifier | Confirm string candidates; classify semantic type, TZ context, clinical event | §TZ, §DOB/DOD, §Validation |
| 8 – Rule Engine | One function per rule group | All |
| 9 – Write Results | Append findings to results Delta table | — |
| 10 – Summary | Blocking findings first, then ranked tables & columns | §Validation, §Enforcement |

> **Hard constraint:** `UPDATE`, `MERGE`, `DELETE`, `CTAS` on source tables are never executed.
> **Scope:** All DATE, TIMESTAMP, TIMESTAMP_NTZ columns are always screened. STRING columns are screened if they look like dates by name or sample content.


# MOSAIC Date / Datetime Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## How to read a finding

| Field | What it means |
|---|---|
| `detected_format` | The actual storage format found (DATE, TIMESTAMP, MM/DD/YYYY, epoch_seconds, etc.) |
| `semantic_type` | What the column represents (date_of_birth, date_of_death, open_ended_end_date, clinical_event, datetime_with_tz, date_only, other) |
| `timezone_context` | The timezone context inferred from column name or AI (_est, _utc, Eastern, UTC, unknown, not_applicable) |
| `has_est_companion` | Whether a corresponding `_est` canonical column exists in the same table |
| `has_source_tz_companion` | Whether a `_utc` or source-tz companion column exists in the same table |
| `standardization_rule` | JSON array of transform options; steward picks one and records `decided_action` |
| `decided_action` | Pre-populated when the procedure leaves no ambiguity; otherwise NULL until steward records choice |

**Blocking findings** (`INVALID_DATE_FORMAT`, `INVALID_DATETIME_FORMAT`, `HIGH_PARSE_FAILURE_RATE`, `MISSING_EST_CANONICAL`, `DOB_FUTURE_DATE`, `DOD_FUTURE_DATE`, `DOB_DOD_CHRONOLOGY`, `DOD_SENTINEL_USED_FOR_ALIVE`, `SUSPECT_SENTINEL_DATE`) appear first in the summary.

---

## Decision workflow

1. Run the notebook — findings written with `decided_action = NULL`.
2. Steward reads `standardization_rule`, picks the right `option_key`, records:
```sql
UPDATE <out_catalog>.<out_schema>.<out_table>
SET    decided_action = '<option_key>', decided_by = '<steward name>'
WHERE  run_id = '<run_id>' AND table_name = '<t>' AND column_name = '<c>' AND classification = '<tag>';
```
3. Engineer queries Section 5 of the summary for their work queue (`decided_action IS NOT NULL`).

---

## Rule definitions

### `INVALID_DATE_FORMAT` ⚠ Blocks certification
**Procedure:** §Format — dates — "Reject or quarantine values that do not parse to valid calendar dates."
**What it checks:** A typed DATE column (or a string column storing dates) contains values that cannot be parsed as valid calendar dates.
**Standardization:** Quarantine non-parseable rows; investigate source system for bad date generation.

---

### `NON_ISO_DATE_FORMAT`
**Procedure:** §Format — dates — "Persist date-only fields in ISO 8601 YYYY-MM-DD format."
**What it checks:** A string column stores date values in a non-ISO format (MM/DD/YYYY, DD-MM-YYYY, epoch seconds, etc.). Values may be valid dates, but the format violates the canonical standard.
**Standardization:** Parse from detected format and reformat to YYYY-MM-DD in the Silver transform. Store format mapping in the rule repository.

---

### `INVALID_DATETIME_FORMAT` ⚠ Blocks certification
**Procedure:** §Format — datetimes — "Reject or quarantine values that do not parse."
**What it checks:** A TIMESTAMP column or string storing datetimes contains unparseable values.
**Standardization:** Quarantine affected rows; investigate source.

---

### `NON_ISO_DATETIME_FORMAT`
**Procedure:** §Format — datetimes — "Persist datetimes using ISO 8601 combined form."
**What it checks:** String column stores datetimes in a non-standard format (not YYYY-MM-DD HH:MM:SS[.fff]).
**Standardization:** Parse and reformat to ISO 8601 in the Silver transform.

---

### `MISSING_EST_CANONICAL` ⚠ May block certification
**Procedure:** §Timezone normalization — "Every datetime model must have a `_est` canonical column."
**What it checks:** A datetime column exists in a table but no corresponding `_est` suffixed column is present. If the column itself is not already named `*_est`, the Eastern canonical is absent.
**Standardization:** Add the `_est` column derived from a timezone conversion in the Silver transform. Apply DST rules per pipeline config.

---

### `MISSING_SOURCE_TZ_COMPANION`
**Procedure:** §Timezone normalization — "When the source is not already US Eastern, retain the original-timezone value in a deterministically named companion column (_utc or source-tz suffix)."
**What it checks:** An `_est` canonical exists but no `_utc` (or other source-tz suffixed) companion column is found, and the source is inferred to be non-Eastern (UTC, etc.).
**Standardization:** Add the `_utc` (or appropriate source-tz) companion column preserving the original value.

---

### `INCONSISTENT_TZ_SUFFIX`
**Procedure:** §Timezone normalization — "Use the same suffix pair across every table so consumers can predict column names."
**What it checks:** A table has datetime columns using inconsistent timezone suffix conventions (e.g. some `_est`, some `_eastern`, some `_ET`, no suffix).
**Standardization:** Standardise to `_est` / `_utc` naming across all columns in the table.

---

### `UNDOCUMENTED_TIMEZONE`
**Procedure:** §Documentation — "Data dictionary must list each datetime column: format, timezone, source system clock."
**What it checks:** A datetime column has no recognisable timezone suffix and AI cannot infer the source timezone from column name or table context.
**Standardization:** Add timezone documentation to the data dictionary; add the appropriate suffix to the column name.

---

### `DOB_FUTURE_DATE` ⚠ Quarantine required
**Procedure:** §DOB/DOD — "Values that are merely suspect or implausible — a future DOB — are flagged and quarantined for steward review, **never silently nulled**."
**What it checks:** A DOB column contains a date in the future.
**Standardization:** Quarantine affected rows; steward decides per-case (keep real value, register new known-null sentinel, or correct/reject). Do NOT null silently.

---

### `DOD_FUTURE_DATE` ⚠ Quarantine required
**Procedure:** §DOB/DOD — "DOD < DOB and not in the future; violations are quarantined."
**What it checks:** A DOD column contains a future date.
**Standardization:** Quarantine affected rows. Do NOT null.

---

### `DOB_DOD_CHRONOLOGY` ⚠ Quarantine required
**Procedure:** §DOB/DOD — "DOD ≥ DOB and not in the future; violations are quarantined."
**What it checks:** Cross-column: DOD < DOB is chronologically impossible.
**Standardization:** Quarantine both rows. Do not alter either field value; preserve for investigation.

---

### `DOB_PLAUSIBILITY`
**Procedure:** §DOB/DOD — "Age > 120 or age < 0 — flagged and quarantined for steward review."
**What it checks:** DOB implies an age outside the plausible human range (default 0–120 years, configurable).
**Standardization:** Quarantine for steward review. Preserve source value.

---

### `SUSPECT_SENTINEL_DATE` ⚠ Quarantine required
**Procedure:** §DOB/DOD — "Unregistered suspect dates (1900-01-01, 9999-12-31, etc.) are quarantined for review, **not auto-nulled**."
**What it checks:** An unregistered sentinel date (1900-01-01, 9999-12-31, or others) found in a DOB or DOD column. These are NOT the registered known-null sentinel (1875-05-20).
**Standardization:** Quarantine and flag for steward investigation. Steward decides: keep, register as known-null sentinel, or correct. Never auto-null.

---

### `KNOWN_NULL_SENTINEL_PRESENT`
**Procedure:** §DOB/DOD — "Set DOB/DOD to NULL only when the value matches a registered known-null sentinel (e.g. 1875-05-20 Cobalt empty-date)."
**What it checks:** The registered Cobalt known-null sentinel (1875-05-20) is present in a DOB or DOD column. This IS a legitimate path to NULL for DOB/DOD.
**Standardization:** Convert to NULL; this is the only case where DOB/DOD may be nulled without steward case-by-case review.

---

### `DOD_SENTINEL_USED_FOR_ALIVE` ⚠ Blocks certification
**Procedure:** §DOB/DOD — "DOD stays genuinely nullable where absent (NULL = not deceased); never substitute a future sentinel (2199-01-01, 9999-12-31) to mean 'alive'."
**What it checks:** A DOD column contains 2199-01-01 or 9999-12-31, which would mean "alive" — a policy violation.
**Standardization:** Convert to NULL (living = NULL DOD); deploy fix before certification.

---

### `MAGIC_DATE_DETECTED`
**Procedure:** §Format — datetimes — "Handle null and sentinel dates via Null and missing values rules — do not use magic dates without dictionary entry."
**What it checks:** A non-DOB/DOD date column has a single date appearing in >= `magic_pct`% of non-null rows — likely a system default or undocumented sentinel.
**Standardization:** Investigate source; register in dictionary or convert to NULL/approved sentinel.

---

### `UNAPPROVED_OPEN_ENDED_SENTINEL`
**Procedure:** §Format — "Use 2199-01-01 for open-ended future dates; do not persist 9999-12-31."
**What it checks:** A column classified as an open-ended end-date (effective_to, valid_to, etc.) contains 9999-12-31, which overflows downstream tools.
**Standardization:** Replace with 2199-01-01 (the MOSAIC approved open-ended sentinel).

---

### `HIGH_PARSE_FAILURE_RATE` ⚠ Blocks certification
**Procedure:** §Validation — "Automated checks must verify parse success rate."
**What it checks:** More than `parse_failure_threshold`% (default 5%) of non-null date/datetime values fail to parse.
**Standardization:** Quarantine all parse failures; investigate source system date generation.

---

### `FUTURE_CLINICAL_EVENT`
**Procedure:** §Validation — "No future-dated clinical events where business rules forbid them."
**What it checks:** A clinical event date column (appointment, procedure, admission, discharge, etc.) contains future dates, which are clinically invalid for recorded events.
**Standardization:** Quarantine future-dated clinical event rows; investigate source.

---

## Extensibility
1. Add tag to `TAGS` in Cell 4 with procedure section reference
2. Add severity to `SEVERITY` in Cell 4
3. Add standardization options to `STANDARDIZATION_RULES` in Cell 4
4. Write `_check_<topic>()` in Cell 8 and call it in the main loop


In [0]:
import re, json, uuid, hashlib
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Tuple, Set
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":        _w("catalog",        "data_classification_source"),
    "schema":         _w("schema",         "date_datetime_detector"),
    "table_filter":   _w("table_filter",   ""),
    "skip_views":     _w("skip_views",     "true").strip().lower() == "true",

    # Layer declaration
    "layer":          _w("layer",          "Unknown"),

    # Sampling
    "sample_size":    int(_w("sample_size",    10_000)),
    "seed":           int(_w("seed",           42)),

    # Detection thresholds
    # parse_failure_threshold: % of non-null values that fail to parse before HIGH_PARSE_FAILURE_RATE fires
    "parse_failure_threshold": float(_w("parse_failure_threshold", "5.0")),
    # magic_pct: % frequency threshold for MAGIC_DATE_DETECTED
    "magic_pct":      float(_w("magic_pct",    "1.0")),
    # age_max: maximum plausible age in years for DOB plausibility check
    "age_max":        int(_w("age_max",         120)),
    # string_parse_threshold: % of sample that must parse as date for string candidate detection
    "string_parse_threshold": float(_w("string_parse_threshold", "80.0")),

    # Results
    "out_catalog":    _w("out_catalog",    "data_classification_target"),
    "out_schema":     _w("out_schema",     "data_classification_output"),
    "out_table":      _w("out_table",      "date_findings"),

    # AI
    "ai_model":       _w("ai_model",       "databricks-meta-llama-3-3-70b-instruct"),
    # enable_ai: when True, uses ai_query() for:
    #   (1) string column candidate confirmation
    #   (2) semantic type classification (DOB/DOD/clinical_event/open_ended/other)
    #   (3) timezone context inference
    #   (4) date format pattern identification for non-ISO string columns
    "enable_ai":      _w("enable_ai",      "true").strip().lower() == "true",
}

RUN_ID  = str(uuid.uuid4())
RUN_TS  = datetime.utcnow().isoformat()
TODAY   = date.today()

print(f"Run      : {RUN_ID}")
print(f"Scope    : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer    : {CFG['layer']}")
print(f"Today    : {TODAY}")
print(f"AI       : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results  : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Write _check_<topic>() in Cell 8 and call it in the main loop
# ---------------------------------------------------------------------------

TAGS = {
    # §Format
    "INVALID_DATE_FORMAT":          "§Format-dates",
    "NON_ISO_DATE_FORMAT":          "§Format-dates",
    "INVALID_DATETIME_FORMAT":      "§Format-datetimes",
    "NON_ISO_DATETIME_FORMAT":      "§Format-datetimes",
    "HIGH_PARSE_FAILURE_RATE":      "§Validation",
    "FORMAT_CLASSIFICATION_INCOMPLETE": "§Format-datetimes",
    # §Timezone
    "MISSING_EST_CANONICAL":        "§TZ-normalization",
    "MISSING_SOURCE_TZ_COMPANION":  "§TZ-normalization",
    "INCONSISTENT_TZ_SUFFIX":       "§TZ-normalization",
    "UNDOCUMENTED_TIMEZONE":        "§Documentation",
    # §DOB/DOD
    "DOB_FUTURE_DATE":              "§DOB-preservation",
    "DOD_FUTURE_DATE":              "§DOD-preservation",
    "DOB_DOD_CHRONOLOGY":           "§DOB-DOD",
    "DOB_PLAUSIBILITY":             "§DOB-preservation",
    "SUSPECT_SENTINEL_DATE":        "§DOB-DOD",
    "KNOWN_NULL_SENTINEL_PRESENT":  "§DOB-DOD",
    "DOD_SENTINEL_USED_FOR_ALIVE":  "§DOD-preservation",
    # §Generic sentinels
    "MAGIC_DATE_DETECTED":          "§Format-datetimes",
    "UNAPPROVED_OPEN_ENDED_SENTINEL":"§Format-datetimes",
    # §Validation
    "FUTURE_CLINICAL_EVENT":        "§Validation",
}

SEVERITY: Dict[str, int] = {
    "INVALID_DATE_FORMAT":          10,
    "INVALID_DATETIME_FORMAT":      10,
    "HIGH_PARSE_FAILURE_RATE":      10,
    "FORMAT_CLASSIFICATION_INCOMPLETE": 7,
    "DOB_FUTURE_DATE":               9,
    "DOD_FUTURE_DATE":               9,
    "DOB_DOD_CHRONOLOGY":            9,
    "DOD_SENTINEL_USED_FOR_ALIVE":   9,
    "SUSPECT_SENTINEL_DATE":         8,
    "MISSING_EST_CANONICAL":         7,
    "NON_ISO_DATE_FORMAT":           6,
    "NON_ISO_DATETIME_FORMAT":       6,
    "DOB_PLAUSIBILITY":              6,
    "KNOWN_NULL_SENTINEL_PRESENT":   5,
    "FUTURE_CLINICAL_EVENT":         5,
    "MISSING_SOURCE_TZ_COMPANION":   4,
    "UNAPPROVED_OPEN_ENDED_SENTINEL":4,
    "INCONSISTENT_TZ_SUFFIX":        3,
    "MAGIC_DATE_DETECTED":           3,
    "UNDOCUMENTED_TIMEZONE":         2,
}

# ── Date dtype sets ───────────────────────────────────────────────────────────
DATE_DTYPES:      Set[str] = {"DATE"}
TS_DTYPES:        Set[str] = {"TIMESTAMP", "TIMESTAMP_NTZ", "TIMESTAMP_LTZ"}
STRING_DTYPES_PFX           = ("STRING", "VARCHAR", "CHAR")

# ── Sentinel dates ────────────────────────────────────────────────────────────
DATE_COBALT   = date(1875,  5, 20)   # registered known-null sentinel (may be NULLed)
DATE_EPOCH    = date(1900,  1,  1)   # unregistered suspect -- quarantine in DOB/DOD
DATE_MAX      = date(9999, 12, 31)   # max-date sentinel -- quarantine in DOB/DOD
DATE_MOSAIC   = date(2199,  1,  1)   # MOSAIC approved open-ended (illegal in DOD)
KNOWN_SENTINELS: Set[date] = {DATE_COBALT, DATE_EPOCH, DATE_MAX, DATE_MOSAIC}

# Suspect sentinels in DOB/DOD (must quarantine, not null)
SUSPECT_SENTINELS: List[str] = ["DATE'1900-01-01'", "DATE'9999-12-31'"]

# ── Timezone suffix patterns ─────────────────────────────────────────────────
RE_EST  = re.compile(r"_(est|et|eastern|edt|est5edt)$", re.I)
RE_UTC  = re.compile(r"_(utc|gmt|z)$", re.I)
RE_TZ   = re.compile(r"_(est|et|eastern|edt|est5edt|utc|gmt|z|pst|pt|cst|ct|mst|mt|at|ast)$", re.I)

# ── Column name heuristics ────────────────────────────────────────────────────
# RE_DATE_NAME: column names that suggest a date/datetime field.
# Only include terms that typically NAME A DATE, not terms that name
# an event type (procedure, diagnosis, etc.) -- those can appear in
# name/text columns and would cause false date candidate matches.
RE_DATE_NAME = re.compile(
    r"(date|_dt$|_ts$|_at$|time|timestamp|created|updated|modified|recorded|"
    r"birth|death|dob|dod|start|end|begin|effective|expir|valid|open|close|"
    r"admission|discharge|appointment|visit|encounter|event|"
    r"posted|issued|submitted|processed|completed|cancelled|occurred|reported|"
    r"fecha|datum|dato|ts_|dt_|_date|_time)",
    re.I
)
# Explicit name-field exclusion: these words in a column name suggest a label/category,
# not a date value, even if they appear in RE_DATE_NAME context.
RE_DATE_NAME_EXCLUDE = re.compile(
    r"(_name$|_type$|_code$|_label$|_desc$|_description$|_category$|_status$|"
    r"_reason$|_note$|_comment$|_text$|_title$|_summary$)",
    re.I
)
RE_DOB     = re.compile(r"(dob|date.?of.?birth|birth.?date|birthdate|dt_nasci|fecha_nac)", re.I)
RE_DOD     = re.compile(r"(dod|date.?of.?death|death.?date|deathdate|dt_obit|fecha_def)", re.I)
RE_ENDDATE = re.compile(r"(effective.?to|valid.?to|end.?date|to.?date|expir|termination|coverage.?end|close.?date)", re.I)
RE_ENDDATE_EXCLUDE = re.compile(
    r"^(transaction|event|created|updated|modified|recorded|admission|discharge|"
    r"order|payment|invoice|processed|submitted|posted|issued|started|opened).?date", re.I
)
RE_CLINICAL = re.compile(
    r"(admission|discharge|appointment|procedure|visit|encounter|event|"
    r"surgery|treatment|diagnosis|referral|assessment|sample|collection|result|"
    r"lab|test|exam|consult|session)", re.I
)

# Semantic types
SEMANTIC_TYPES = {
    "date_of_birth", "date_of_death", "open_ended_end_date",
    "clinical_event_date", "datetime_with_tz", "date_only", "other"
}

def _heuristic_semantic(col: str, dtype: str) -> str:
    if RE_DOB.search(col):     return "date_of_birth"
    if RE_DOD.search(col):     return "date_of_death"
    if RE_ENDDATE.search(col) and not RE_ENDDATE_EXCLUDE.search(col):
        return "open_ended_end_date"
    if RE_CLINICAL.search(col): return "clinical_event_date"
    if dtype in TS_DTYPES:     return "datetime_with_tz"
    return "date_only"

def _heuristic_tz(col: str) -> str:
    if RE_EST.search(col): return "_est"
    if RE_UTC.search(col): return "_utc"
    if RE_TZ.search(col):  return "other_tz"
    return "unknown"

# ── Standardization rules ─────────────────────────────────────────────────────
STANDARDIZATION_RULES: Dict[str, list] = {

    "INVALID_DATE_FORMAT": [
        {"option_key": "quarantine_invalid",
         "label":      "Quarantine rows with non-parseable dates",
         "sql":        "-- Route rows WHERE TRY_CAST(col AS DATE) IS NULL AND col IS NOT NULL to quarantine table.",
         "notes":      "Non-parseable dates must not persist in curated layers (§Format-dates). "
                       "Investigate source system. Rejected rows must follow the data product quarantine procedure."},
    ],

    "NON_ISO_DATE_FORMAT": [
        {"option_key": "reformat_to_iso",
         "label":      "Parse from detected format and reformat to YYYY-MM-DD in Silver",
         "sql":        "-- SILVER: TO_DATE(col, '<detected_format>') AS col  -- replace <detected_format>",
         "notes":      "Store format mapping in the rule repository. "
                       "Bronze preserves source format. Silver must use ISO 8601 (§Format-dates)."},
        {"option_key": "investigate_format",
         "label":      "Investigate format before applying transform",
         "sql":        "-- No transform until format is confirmed and mapping is documented.",
         "notes":      "Use when AI format detection has low confidence. "
                       "Confirm format with source system owner before adding to rule repository."},
    ],

    "INVALID_DATETIME_FORMAT": [
        {"option_key": "quarantine_invalid",
         "label":      "Quarantine rows with non-parseable datetimes",
         "sql":        "-- Typed TIMESTAMP: WHERE TRY_CAST(col AS TIMESTAMP) IS NULL AND col IS NOT NULL\n"
                       "-- STRING column:   WHERE TRY_TO_TIMESTAMP(col, '<detected_format>') IS NULL AND col IS NOT NULL\n"
                       "-- Route matching rows to quarantine table.",
         "notes":      "Non-parseable datetimes must not persist in curated layers (§Format-datetimes). "
                       "For STRING columns use TRY_TO_TIMESTAMP with the confirmed detected_format "
                       "(see standardization_rule on the accompanying NON_ISO_DATETIME_FORMAT finding); "
                       "for typed TIMESTAMP columns use TRY_CAST. Investigate source system."},
    ],

    "FORMAT_CLASSIFICATION_INCOMPLETE": [
        {"option_key": "manual_format_determination",
         "label":      "Manually determine format, then re-run classification or apply transform directly",
         "sql":        "-- Inspect sample_values; confirm actual source format with the source system owner.",
         "notes":      "AI format classification did not resolve a concrete date_format for this "
                       "confirmed date-like STRING column (Cell 5 AI query failed or omitted it). "
                       "Do not apply a Silver reformat transform using a placeholder format -- "
                       "confirm the real format first, then re-run classification (§Format)."},
        {"option_key": "rerun_classification",
         "label":      "Re-run the AI classifier for this table",
         "sql":        "-- Re-run Cell 5 (AI Date Classifier) for this table/column.",
         "notes":      "Use when the failure looks transient (timeout, rate limit) rather than a genuinely ambiguous format."},
    ],

    "NON_ISO_DATETIME_FORMAT": [
        {"option_key": "reformat_to_iso",
         "label":      "Parse and reformat to ISO 8601 YYYY-MM-DD HH:MM:SS[.fff] in Silver",
         "sql":        "-- SILVER: TO_TIMESTAMP(col, '<detected_format>') AS col",
         "notes":      "Canonical datetime format is ISO 8601 combined form (§Format-datetimes). "
                       "Document fractional-second precision (zzz=milliseconds when needed)."},
        {"option_key": "investigate_format",
         "label":      "Investigate format before applying transform",
         "sql":        "-- No transform until format is confirmed.",
         "notes":      "Use when format detection confidence is low."},
    ],

    "HIGH_PARSE_FAILURE_RATE": [
        {"option_key": "quarantine_failures",
         "label":      "Quarantine all parse failures; investigate source",
         "sql":        "-- Route rows WHERE TRY_CAST(col AS DATE/TIMESTAMP) IS NULL AND col IS NOT NULL to quarantine.",
         "notes":      "Parse success rate must be verified by automated DQ checks (§Validation). "
                       "Failed records follow the data product quarantine or reject procedure."},
    ],

    "MISSING_EST_CANONICAL": [
        {"option_key": "add_est_column",
         "label":      "Add _est canonical column via timezone conversion in Silver transform",
         "sql":        "-- SILVER: CONVERT_TIMEZONE('US/Eastern', col) AS <col>_est",
         "notes":      "Every datetime model must have a _est canonical column (§TZ-normalization). "
                       "Apply DST rules explicitly in pipeline config. "
                       "If source is already Eastern, cast is: col AS <col>_est (no conversion)."},
    ],

    "MISSING_SOURCE_TZ_COMPANION": [
        {"option_key": "add_source_tz_companion",
         "label":      "Add source-timezone companion column (e.g. _utc) preserving original value",
         "sql":        "-- SILVER: col AS <col>_utc  -- preserve original; do not overwrite",
         "notes":      "When source is non-Eastern, preserve original in <col>_utc (§TZ-normalization). "
                       "Never overwrite or discard the source value. "
                       "Record source timezone and conversion logic in the data mapping artifact."},
    ],

    "INCONSISTENT_TZ_SUFFIX": [
        {"option_key": "standardise_suffix",
         "label":      "Standardise to _est / _utc naming convention across all columns",
         "sql":        "-- Rename columns to use _est (Eastern canonical) and _utc (source-tz original).",
         "notes":      "Use the same suffix pair across every table so consumers can predict column names "
                       "(§TZ-normalization). Renaming requires downstream BI contract review."},
    ],

    "UNDOCUMENTED_TIMEZONE": [
        {"option_key": "document_and_suffix",
         "label":      "Add timezone to data dictionary and apply standard suffix",
         "sql":        "-- Add _est or _utc suffix to column name; document timezone in data dictionary.",
         "notes":      "Data dictionary must list timezone and source system clock (§Documentation). "
                       "Apply standard suffix convention after confirming source timezone."},
        {"option_key": "investigate_source_tz",
         "label":      "Investigate source timezone before documenting",
         "sql":        "-- No transform until source timezone is confirmed with the source system owner.",
         "notes":      "Use when AI cannot determine source timezone from context."},
    ],

    "DOB_FUTURE_DATE": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows with future DOB for steward review -- do NOT null",
         "sql":        "-- Route rows WHERE dob > CURRENT_DATE to quarantine table.",
         "notes":      "Preservation override: future DOB must be quarantined, never silently nulled (§DOB-preservation). "
                       "Steward decides per case: keep real value, register new known-null sentinel, or reject."},
    ],

    "DOD_FUTURE_DATE": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows with future DOD for steward review -- do NOT null",
         "sql":        "-- Route rows WHERE dod > CURRENT_DATE to quarantine table.",
         "notes":      "Future DOD is clinically impossible. Quarantine; do not null (§DOD-preservation). "
                       "Steward investigates source."},
    ],

    "DOB_DOD_CHRONOLOGY": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows where DOD < DOB (chronologically impossible)",
         "sql":        "-- Route rows WHERE dod < dob to quarantine table. Preserve both field values.",
         "notes":      "Do not alter either field -- preserve source values for investigation (§DOB-DOD). "
                       "Exclude from EMPI matching and mortality reporting until resolved."},
    ],

    "DOB_PLAUSIBILITY": [
        {"option_key": "quarantine",
         "label":      "Quarantine implausible DOB rows for steward review -- do NOT null",
         "sql":        f"-- Route rows WHERE derived_age < 0 OR derived_age > {CFG.get('age_max', 120)} to quarantine.",
         "notes":      "Preservation override: implausible DOB is quarantined, never nulled (§DOB-preservation). "
                       "Steward reviews each case. Preserve source value in quarantine."},
    ],

    "SUSPECT_SENTINEL_DATE": [
        {"option_key": "quarantine_pending_investigation",
         "label":      "Quarantine rows with suspect sentinel for steward investigation -- do NOT null",
         "sql":        "-- Route rows WHERE dob/dod IN (DATE'1900-01-01', DATE'9999-12-31') to quarantine.",
         "notes":      "Preservation override: unregistered sentinels in DOB/DOD are quarantined, not auto-nulled (§DOB-DOD). "
                       "Steward decides: keep value, register as known-null sentinel, or correct/reject. "
                       "Only registered sentinels (1875-05-20) may be auto-nulled."},
    ],

    "KNOWN_NULL_SENTINEL_PRESENT": [
        {"option_key": "convert_to_null",
         "label":      "Convert registered known-null sentinel (1875-05-20) to NULL",
         "sql":        "CASE WHEN col = DATE'1875-05-20' THEN NULL ELSE col END",
         "notes":      "1875-05-20 is the Cobalt system empty-date -- a registered known-null sentinel. "
                       "This is the ONLY case where DOB/DOD may be auto-nulled (§DOB-DOD). "
                       "Register in data dictionary before deploying."},
    ],

    "DOD_SENTINEL_USED_FOR_ALIVE": [
        {"option_key": "convert_to_null",
         "label":      "Convert sentinel to NULL -- living patients must have NULL DOD",
         "sql":        "CASE WHEN dod IN (DATE'2199-01-01', DATE'9999-12-31') THEN NULL ELSE dod END",
         "notes":      "POLICY VIOLATION: living = NULL DOD, never a future sentinel (§DOD-preservation). "
                       "Blocks certification. Deploy fix before republication."},
    ],

    "MAGIC_DATE_DETECTED": [
        {"option_key": "investigate_and_register",
         "label":      "Investigate the date; register in dictionary or convert to NULL/sentinel",
         "sql":        "-- CASE WHEN col = DATE'<confirmed_sentinel>' THEN NULL ELSE col END",
         "notes":      "Disproportionately frequent date suggests a system default or undocumented sentinel. "
                       "Do not use magic dates without a dictionary entry (§Format-datetimes). "
                       "If confirmed: register and apply transform. If unconfirmed: preserve and monitor."},
        {"option_key": "preserve_pending_review",
         "label":      "Preserve as-is pending steward investigation",
         "sql":        "-- No transform until magic date meaning is confirmed.",
         "notes":      "Use when source system context is unclear."},
    ],

    "UNAPPROVED_OPEN_ENDED_SENTINEL": [
        {"option_key": "replace_with_mosaic_sentinel",
         "label":      "Replace 9999-12-31 with MOSAIC approved sentinel 2199-01-01",
         "sql":        "CASE WHEN col = DATE'9999-12-31' THEN DATE'2199-01-01' ELSE col END",
         "notes":      "9999-12-31 overflows downstream tools; use 2199-01-01 for open-ended ranges (§Format-datetimes). "
                       "Requires steward confirmation."},
    ],

    "FUTURE_CLINICAL_EVENT": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows with future clinical event date",
         "sql":        "-- Route rows WHERE col > CURRENT_DATE to quarantine table.",
         "notes":      "Recorded clinical events cannot occur in the future (§Validation). "
                       "Investigate source system for erroneous future dates."},
        {"option_key": "preserve_if_scheduled",
         "label":      "Preserve if column stores scheduled/planned events (not recorded events)",
         "sql":        "-- No transform. Document in data dictionary that column stores scheduled dates.",
         "notes":      "Use when the column stores future-scheduled dates (appointments, procedures). "
                       "Document this clearly in the data dictionary to distinguish from recorded events."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema.
# Builds: tables, table_cols, table_col_names, table_col_names_lower
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    """Escape backticks in catalog/schema/table/column names for SQL."""
    return name.replace("`", "``")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_cols[r.table_name].append((r.column_name, r.data_type.upper()))

# Pre-build lowercased column name sets once -- used in _tz_companions() for O(1) lookup
table_col_names:       Dict[str, Set[str]] = {tbl: {col for col, _ in cols} for tbl, cols in table_cols.items()}
table_col_names_lower: Dict[str, Set[str]] = {tbl: {col.lower() for col in names} for tbl, names in table_col_names.items()}

total_cols = sum(len(v) for v in table_cols.values())
print(f"Scope   : {cat}.{sch}")
print(f"Tables  : {len(tables)}")
print(f"Columns : {total_cols}")
print(f"Layer   : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# SCALABLE PROFILER -- key design decisions:
#
# 1. ONE SQL per table for ALL aggregate stats (null counts, sentinel counts,
#    parse failures, future dates, DOB plausibility). No per-column queries.
#
# 2. sample_df.cache() after collection; unpersist() after table is done.
#
# 3. Column name escaping via _esc() for all SQL references.
#
# 4. String date candidates screened from the cached sample DF only --
#    no additional full table scans during candidate detection.
#
# 5. _tz_companions() uses pre-built table_col_names_lower for O(1) lookup.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def _is_date(dt: str) -> bool:  return dt in DATE_DTYPES
def _is_ts(dt: str) -> bool:    return dt in TS_DTYPES
def _is_str(dt: str) -> bool:   return any(dt.startswith(p) for p in STRING_DTYPES_PFX)

def mask(values: list, n: int = 5) -> list:
    return [str(v) for v in values if v is not None][:n]


def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`")
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def _tz_companions(tbl: str, col: str) -> Tuple[bool, bool, str]:
    """O(1) TZ companion lookup using pre-built lowercase sets from discovery."""
    col_lower  = col.lower()
    names_low  = table_col_names_lower.get(tbl, set())

    if RE_EST.search(col):
        base = RE_EST.sub("", col_lower)
        has_utc = f"{base}_utc" in names_low or f"{base}_gmt" in names_low
        return True, has_utc, "_est"

    if RE_UTC.search(col):
        base = RE_UTC.sub("", col_lower)
        has_est = f"{base}_est" in names_low or f"{base}_et" in names_low
        return has_est, True, "_utc"

    if RE_TZ.search(col):
        return False, False, "other_tz"

    has_est = f"{col_lower}_est" in names_low
    has_utc = f"{col_lower}_utc" in names_low
    return has_est, has_utc, "no_suffix"


# Non-ISO formats commonly seen in vendor/legacy feeds. Tried in addition to
# ISO parsing so that a legitimately date-shaped column is never scored at a
# 0% parse rate just because its format isn't ISO -- that 0% used to cause
# name-matched date columns to be dropped before the AI classifier ever saw
# them. This list only needs to be good enough to produce a non-zero parse
# rate signal for the AI classifier -- Cell 5 (AI Date Classifier) still makes
# the authoritative is_date / date_format call from actual sample values.
_STRING_DATE_PATTERNS = [
    "yyyy-MM-dd", "yyyy-MM-dd HH:mm:ss", "yyyy/MM/dd",
    "MM/dd/yyyy", "M/d/yyyy", "dd/MM/yyyy", "d/M/yyyy",
    "MM-dd-yyyy", "dd-MM-yyyy", "dd.MM.yyyy",
    "yyyyMMdd", "dd-MMM-yyyy", "MMM d, yyyy", "MMMM d, yyyy",
]

def screen_string_col(col: str, sample_df: DataFrame) -> Tuple[bool, float]:
    """
    Estimate what fraction of non-null values in a STRING column parse as dates.
    Runs on the CACHED sample DF only -- no full table scan.
    Tries ISO parsing plus a set of common non-ISO patterns (_STRING_DATE_PATTERNS)
    so vendor formats like MM/DD/YYYY or YYYYMMDD register a real parse rate
    instead of a false 0%. Also falls back to the generic to_timestamp() parser.
    Returns (is_candidate, parse_rate_pct).
    """
    name_match = bool(RE_DATE_NAME.search(col)) and not bool(RE_DATE_NAME_EXCLUDE.search(col))
    try:
        v = F.col("v")
        parsed_cond = F.to_timestamp(v).isNotNull()
        for pat in _STRING_DATE_PATTERNS:
            parsed_cond = parsed_cond | F.to_date(v, pat).isNotNull()

        row = (
            sample_df
            .select(F.col(f"`{_esc(col)}`").alias("v"))
            .filter(F.col("v").isNotNull() & (F.trim(F.col("v")) != ""))
            .agg(
                F.count("*").alias("total"),
                F.count(F.when(parsed_cond, True)).alias("parsed")
            )
            .collect()[0]
        )
        total  = row["total"] or 0
        parsed = row["parsed"] or 0
        rate   = parsed / total * 100 if total > 0 else 0.0
        is_cand = name_match or rate >= CFG["string_parse_threshold"]
        return is_cand, rate
    except Exception:
        return name_match, 0.0


def profile_table_batch(tbl: str, candidates: Dict[str, dict]) -> Dict[str, dict]:
    """
    ONE SQL per table collecting ALL aggregate stats for ALL candidate columns.
    Includes: null_count, parse_fail, sentinel counts, future dates, plausibility.
    Returns updated candidate dict with stats merged in.
    """
    if not candidates:
        return candidates

    fq   = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    for col, info in candidates.items():
        dtype = info["dtype"]
        cs    = f"`{_esc(col)}`"
        safe  = col.replace("`","").replace(" ","_").replace(".","_")[:60]

        # Cast expression.
        # NOTE: for STRING columns, detected_format is NOT yet known here --
        # AI format classification (Step 2 in Cell 5) runs AFTER this batch
        # profile. The previous ISO-only cast (TRY_CAST ... AS DATE/TIMESTAMP)
        # meant every stat computed below (parse_fail, sentinel counts, future
        # dates, DOB plausibility) was wrong for any non-ISO string column:
        # legitimate MM/DD/YYYY-style values registered as parse failures,
        # while genuinely unparseable values (e.g. "NOT_A_DATE") could be
        # miscounted depending on which branch matched. Try ISO plus every
        # pattern in _STRING_DATE_PATTERNS (same list used by screen_string_col
        # above) so these stats reflect the column's actual format.
        if _is_date(dtype):
            cast = f"TRY_CAST({cs} AS DATE)"
        elif _is_ts(dtype):
            cast = f"TRY_CAST({cs} AS TIMESTAMP)"
        else:
            fmt_attempts = ", ".join(f"TRY_TO_DATE({cs}, '{p}')" for p in _STRING_DATE_PATTERNS)
            cast = (f"COALESCE(TRY_CAST({cs} AS DATE), "
                    f"TRY_CAST(TRY_CAST({cs} AS TIMESTAMP) AS DATE), {fmt_attempts})")

        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND {cast} IS NULL THEN 1 END) AS `__fail__{safe}`",
            f"COUNTIF(TRY_CAST({cast} AS DATE) = DATE'1875-05-20') AS `__cobalt__{safe}`",
            f"COUNTIF(TRY_CAST({cast} AS DATE) = DATE'1900-01-01') AS `__epoch__{safe}`",
            f"COUNTIF(TRY_CAST({cast} AS DATE) = DATE'9999-12-31') AS `__max__{safe}`",
            f"COUNTIF(TRY_CAST({cast} AS DATE) = DATE'2199-01-01') AS `__mosaic__{safe}`",
            f"COUNTIF(TRY_CAST({cast} AS DATE) > DATE'{TODAY}')    AS `__future__{safe}`",
        ]
        # DOB plausibility (age out of range)
        if info.get("semantic_type") == "date_of_birth":
            exprs += [
                f"COUNTIF(TRY_CAST({cast} AS DATE) IS NOT NULL AND "
                f"  DATEDIFF(year, TRY_CAST({cast} AS DATE), DATE'{TODAY}') > {CFG['age_max']}) AS `__old__{safe}`",
                f"COUNTIF(TRY_CAST({cast} AS DATE) IS NOT NULL AND "
                f"  DATEDIFF(year, TRY_CAST({cast} AS DATE), DATE'{TODAY}') < 0) AS `__young__{safe}`",
            ]

    try:
        row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Batch profile failed for {tbl}: {e}")
        return candidates

    total = row["__total__"] or 0

    for col, info in candidates.items():
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        info.update({
            "total":       total,
            "null_count":  row.get(f"__null__{safe}", 0) or 0,
            "parse_fail":  row.get(f"__fail__{safe}", 0) or 0,
            "c_cobalt":    row.get(f"__cobalt__{safe}", 0) or 0,
            "c_epoch":     row.get(f"__epoch__{safe}", 0) or 0,
            "c_max":       row.get(f"__max__{safe}", 0) or 0,
            "c_mosaic":    row.get(f"__mosaic__{safe}", 0) or 0,
            "c_future":    row.get(f"__future__{safe}", 0) or 0,
            "c_old":       row.get(f"__old__{safe}", 0) or 0,
            "c_young":     row.get(f"__young__{safe}", 0) or 0,
        })

    return candidates


def magic_date_check(tbl: str, col: str, info: dict, sample_df: DataFrame) -> List[Tuple]:
    """
    Find disproportionately frequent dates. Runs on the sample DF (not full table)
    to avoid an additional full scan. Returns list of (date_val, count) tuples.
    """
    non_null = info["total"] - info["null_count"]
    if non_null == 0:
        return []
    threshold_pct = CFG["magic_pct"] / 100
    dtype = info["dtype"]
    cs    = f"`{_esc(col)}`"

    if _is_date(dtype):    cast_expr = F.col(cs)
    elif _is_ts(dtype):    cast_expr = F.to_date(cs)
    else:                  cast_expr = F.coalesce(F.to_date(cs, "yyyy-MM-dd"), F.to_date(F.to_timestamp(cs)))

    try:
        rows = (
            sample_df
            .select(cast_expr.alias("dv"))
            .filter(F.col("dv").isNotNull())
            .groupBy("dv")
            .count()
            .collect()
        )
        sample_total = sum(r["count"] for r in rows)
        threshold = max(1, int(sample_total * threshold_pct))
        return [
            (r["dv"], r["count"])
            for r in rows
            if r["count"] >= threshold
            # NOTE: KNOWN_SENTINELS filter removed. Previously 9999-12-31 and
            # 2199-01-01 were excluded here on the assumption that dedicated
            # sentinel rules would catch them. That assumption only holds for
            # DOB/DOD and open_ended_end_date semantic types -- for generic
            # columns (coverage effective_to, plan end dates) classified as
            # date_only or other, the dedicated rules never fire and these
            # values would disappear from the report entirely. magic_date_check
            # is only called from _check_generic_sentinels which already skips
            # DOB/DOD columns, so no duplicate findings can arise.
        ]
    except Exception:
        return []


def dod_before_dob_sample_check(tbl: str, dod_col: str, dob_col: str,
                                  dod_info: dict, dob_info: dict,
                                  sample_df: DataFrame) -> int:
    """
    Cross-column DOD < DOB check on the sample DF first.
    Returns violation count from the sample. If > 0, caller can promote to full count.
    """
    try:
        dod_dtype = dod_info["dtype"]
        dob_dtype = dob_info["dtype"]
        dod_c = F.col(f"`{_esc(dod_col)}`") if _is_date(dod_dtype) or _is_ts(dod_dtype) else F.to_date(f"`{_esc(dod_col)}`")
        dob_c = F.col(f"`{_esc(dob_col)}`") if _is_date(dob_dtype) or _is_ts(dob_dtype) else F.to_date(f"`{_esc(dob_col)}`")
        count = (
            sample_df
            .filter(dod_c.isNotNull() & dob_c.isNotNull() & (dod_c < dob_c))
            .count()
        )
        return count
    except Exception:
        return 0


# ── Run profiler ──────────────────────────────────────────────────────────────
table_samples:   Dict[str, DataFrame]       = {}
date_candidates: Dict[str, Dict[str, dict]] = defaultdict(dict)
total_screened   = 0
total_candidates = 0

for tbl in tables:
    cols      = table_cols[tbl]
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    # Pass 1: identify candidates from dtype and string screening
    for col, dtype in cols:
        total_screened += 1
        is_cand   = False
        parse_rate = 100.0
        needs_ai  = False

        if _is_date(dtype) or _is_ts(dtype):
            is_cand          = True
            candidate_reason = "dtype"
        elif _is_str(dtype):
            # Skip obvious non-date name patterns before any Spark work
            if RE_DATE_NAME_EXCLUDE.search(col):
                continue
            is_cand, parse_rate = screen_string_col(col, sample_df)
            candidate_reason = "name_heuristic" if RE_DATE_NAME.search(col) else "string_parse_rate"
            needs_ai = True
            # NOTE: earlier versions dropped name-heuristic candidates here when
            # parse_rate == 0.0. Removed: screen_string_col's pattern list can
            # never cover every source format, so a column whose NAME clearly
            # indicates a date (e.g. admission_date) must still be handed to
            # the AI classifier in Cell 5, which confirms/rejects from actual
            # sample values instead of a fixed regex/format list. A genuinely
            # non-date name-matched column (false positive) is simply rejected
            # by the AI classifier and removed from date_candidates there.

        if not is_cand:
            continue

        has_est, has_utc, tz_ctx = _tz_companions(tbl, col)

        try:
            sample_vals = [
                r["v"] for r in
                sample_df
                .select(F.col(f"`{_esc(col)}`").cast("string").alias("v"))
                .filter(F.col("v").isNotNull())
                .limit(5)
                .collect()
            ]
        except Exception:
            sample_vals = []

        sem = _heuristic_semantic(col, dtype)
        date_candidates[tbl][col] = {
            "dtype":                  dtype,
            "total":                  0,        # filled by batch profile below
            "null_count":             0,
            "parse_fail":             0,
            "c_cobalt": 0, "c_epoch": 0, "c_max": 0,
            "c_mosaic": 0, "c_future": 0, "c_old": 0, "c_young": 0,
            "string_parse_rate":      parse_rate,
            "candidate_reason":       candidate_reason,
            "needs_ai_confirm":       needs_ai,
            "has_est_companion":      has_est,
            "has_source_tz_companion":has_utc,
            "timezone_context":       tz_ctx,
            "sample_vals":            sample_vals,
            "semantic_type":          sem,
            "semantic_source":        "heuristic",
            "semantic_confidence":    "medium",
            "detected_format":        dtype if not _is_str(dtype) else "string_date_unknown",
        }
        total_candidates += 1

    # Pass 2: ONE SQL per table to collect ALL aggregate stats
    if date_candidates[tbl]:
        date_candidates[tbl] = profile_table_batch(tbl, date_candidates[tbl])



print(f"Profiler done.")
print(f"  Screened  : {total_screened} columns across {len(tables)} table(s)")
print(f"  Candidates: {total_candidates} date/datetime columns")
for tbl, cols in date_candidates.items():
    ai_needed = sum(1 for c in cols.values() if c["needs_ai_confirm"])
    print(f"  {tbl}: {len(cols)} candidate(s), {ai_needed} need AI confirmation")


In [0]:
# ---------------------------------------------------------------------------
# AI DATE CLASSIFIER -- scalable batch design:
#
# All ai_query() calls are chunked to max BATCH_SIZE columns per call to avoid
# token limit failures on wide tables.
#
# Three responsibilities per table:
# 1. STRING CANDIDATE CONFIRMATION (chunked, one batch per BATCH_SIZE cols)
# 2. SEMANTIC + FORMAT + TZ CLASSIFICATION (chunked)
# 3. SOURCE TIMEZONE INFERENCE (one call per table)
# ---------------------------------------------------------------------------

BATCH_SIZE = 20  # max columns per ai_query() call

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)


def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_table(tbl: str) -> None:
    cands = date_candidates.get(tbl, {})
    if not cands:
        return

    str_candidates = {col: info for col, info in cands.items() if info["needs_ai_confirm"]}
    confirmed_cols = {col for col, info in cands.items() if not info["needs_ai_confirm"]}

    # ── Step 1: Confirm string candidates (chunked) ───────────────────────────
    if str_candidates and CFG["enable_ai"]:
        for chunk_items in _chunked(list(str_candidates.items())):
            payload = json.dumps([{
                "col":        col,
                "samples":    info["sample_vals"][:6],
                "parse_rate": info["string_parse_rate"],
                "reason":     info["candidate_reason"],
            } for col, info in chunk_items], default=str)

            prompt = (
                "Strict data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
                f"Table: {tbl}. Determine if each STRING column stores date or datetime values. "
                "Consider column name (any language/convention), sample values, parse rate. "
                "A column is a date column if it stores calendar dates or timestamps, not labels/codes/names. "
                f"Columns: {payload}. "
                'Return: [{"col":"<n>","is_date":true|false,"date_or_datetime":"date|datetime|unknown",'
                '"confidence":"high|medium|low"}]'
            )
            try:
                for item in json.loads(_ai_query(prompt)):
                    col = item.get("col", "")
                    if col not in str_candidates:
                        continue
                    if item.get("is_date", False):
                        confirmed_cols.add(col)
                        date_candidates[tbl][col]["confirmed"]        = True
                        date_candidates[tbl][col]["date_or_datetime"] = item.get("date_or_datetime","unknown")
                        date_candidates[tbl][col]["ai_confirm_conf"]  = item.get("confidence","low")
                    else:
                        date_candidates[tbl][col]["confirmed"] = False
            except Exception as e:
                print(f"  [WARN] AI confirm chunk failed for {tbl}: {e}")
                # Apply the same retention rules as the post-Step-1 removal loop:
                # keep name_heuristic candidates unconditionally, and keep
                # string_parse_rate candidates whose measured parse rate >= 50%
                # (strong empirical signal that the column IS a date field even
                # if the AI couldn't confirm it). Previously this fallback only
                # kept name_heuristic, so a column like result_date (high parse
                # rate, no name match) was dropped here whenever ai_query()
                # itself errored -- never reaching the removal loop where the
                # 50% floor fix lived.
                for col, info in chunk_items:
                    reason     = info.get("candidate_reason")
                    parse_rate = info.get("string_parse_rate", 0.0)
                    keep = (reason == "name_heuristic" or
                            (reason == "string_parse_rate" and parse_rate >= 50.0))
                    date_candidates[tbl][col]["confirmed"] = keep
                    if keep:
                        date_candidates[tbl][col]["ai_flagged_not_date"] = True
                        confirmed_cols.add(col)
    else:
        for col in str_candidates:
            date_candidates[tbl][col]["confirmed"] = True
            confirmed_cols.add(col)
        for col in [c for c, i in cands.items() if not i["needs_ai_confirm"]]:
            date_candidates[tbl][col]["confirmed"] = True

    # Remove rejected candidates -- EXCEPT name-matched columns.
    # A column name like "result_date" or "admission_date" is a strong,
    # deterministic signal on its own. If the AI's is_date judgment (made from
    # only ~6 sample values) flips to False -- e.g. because one dirty/garbage
    # value like "NOT_A_DATE" sits among otherwise-valid dates -- deleting the
    # column here would remove it from EVERY downstream check, including the
    # deterministic parse-failure-rate and format checks in the Rule Engine.
    # That silently exempts exactly the messiest columns (the ones most in
    # need of a HIGH_PARSE_FAILURE_RATE / quarantine finding) from the report.
    # Only "string_parse_rate"-reason candidates (no name signal, included
    # purely because some values happened to parse) are dropped on AI
    # rejection -- for those, the AI genuinely is the tie-breaker.
    for col in list(date_candidates[tbl].keys()):
        info = date_candidates[tbl][col]
        if info.get("confirmed", True):
            continue
        reason     = info.get("candidate_reason")
        parse_rate = info.get("string_parse_rate", 0.0)
        # name_heuristic: the column name is a deterministic date signal --
        # keep regardless of AI verdict (AI sees only ~6 samples, a single
        # dirty value can flip its judgment on an otherwise-valid date column).
        if reason == "name_heuristic":
            info["confirmed"] = True
            info["ai_flagged_not_date"] = True
            confirmed_cols.add(col)
            continue
        # string_parse_rate: no name signal, but the profiler measured that
        # >= 50% of sampled values parsed as some date format. That's a strong
        # empirical signal that the column IS a date field with dirty rows --
        # precisely the situation that needs a HIGH_PARSE_FAILURE_RATE finding.
        # Dropping it here on an AI rejection would silence that finding for
        # the messiest columns. Keep it; tag for transparency.
        if reason == "string_parse_rate" and parse_rate >= 50.0:
            info["confirmed"] = True
            info["ai_flagged_not_date"] = True
            confirmed_cols.add(col)
            continue
        del date_candidates[tbl][col]
        confirmed_cols.discard(col)

    if not confirmed_cols:
        return

    # ── Step 2: Semantic + format + TZ classification (chunked) ──────────────
    if CFG["enable_ai"]:
        for chunk in _chunked(list(confirmed_cols)):
            payload = json.dumps([{
                "col":     col,
                "dtype":   date_candidates[tbl][col]["dtype"],
                "samples": date_candidates[tbl][col]["sample_vals"][:6],
                "tz_ctx":  date_candidates[tbl][col]["timezone_context"],
            } for col in chunk if col in date_candidates[tbl]], default=str)

            prompt = (
                "Data classification expert. Reply ONLY with a JSON array -- no prose, no markdown. "
                f"Table: {tbl}. For each date/datetime column classify: "
                "(1) semantic_type: date_of_birth, date_of_death, open_ended_end_date, "
                "clinical_event_date (recorded past events: admission, discharge, appointment, "
                "procedure, lab, diagnosis, visit -- cannot be future), datetime_with_tz, date_only, other. "
                "open_ended_end_date = range BOUNDARY only (effective_to, valid_to, expiry). "
                "Point-in-time events are NOT open_ended_end_date. DOB/DOD override all other types. "
                "(2) date_format: YYYY-MM-DD, YYYY-MM-DD_HH:MM:SS, MM/DD/YYYY, DD/MM/YYYY, "
                "DD-MM-YYYY, DD.MM.YYYY, YYYY/MM/DD, YYYYMMDD, epoch_seconds, epoch_millis, "
                "typed_date, typed_timestamp, other_format. "
                "(3) source_tz: Eastern, UTC, Pacific, Central, Mountain, local_unknown, not_applicable. "
                "(4) confidence: high/medium/low. "
                f"Columns: {payload}. "
                'Return: [{"col":"<n>","semantic_type":"<t>","date_format":"<f>",'
                '"source_tz":"<tz>","confidence":"high|medium|low"}]'
            )
            try:
                for item in json.loads(_ai_query(prompt)):
                    col = item.get("col", "")
                    if col not in confirmed_cols or col not in date_candidates[tbl]:
                        continue
                    sem  = item.get("semantic_type", "other")
                    fmt  = item.get("date_format", "unknown")
                    stz  = item.get("source_tz", "unknown")
                    conf = item.get("confidence", "low")
                    h_sem = date_candidates[tbl][col]["semantic_type"]
                    # DOB/DOD heuristic always wins
                    if h_sem in ("date_of_birth", "date_of_death"):
                        sem = h_sem
                    date_candidates[tbl][col].update({
                        "semantic_type":       sem if sem in SEMANTIC_TYPES else "other",
                        "semantic_source":     "both" if h_sem != "other" else "ai",
                        "semantic_confidence": conf,
                        "detected_format":     fmt,
                        "source_tz":           stz,
                    })
            except Exception as e:
                print(f"  [WARN] AI semantic chunk failed for {tbl}: {e}")
                # Mark every column in this chunk so the Rule Engine (Cell 6)
                # can raise FORMAT_CLASSIFICATION_INCOMPLETE instead of silently
                # leaving detected_format at the profiler's placeholder value
                # ("string_date_unknown") -- which previously got misreported
                # as a concrete NON_ISO_* finding with an unusable format.
                for col in chunk:
                    if col in date_candidates[tbl]:
                        date_candidates[tbl][col]["ai_classification_error"] = str(e)

    # ── Step 3: Source TZ inference for tables with unknown TZ ───────────────
    no_tz = [col for col in confirmed_cols
              if col in date_candidates[tbl]
              and date_candidates[tbl][col].get("timezone_context") == "no_suffix"
              and date_candidates[tbl][col].get("dtype") in TS_DTYPES]

    if no_tz and CFG["enable_ai"]:
        payload = json.dumps([{
            "col":     col,
            "samples": date_candidates[tbl][col]["sample_vals"][:4],
        } for col in no_tz[:10]], default=str)
        prompt = (
            "Timezone inference expert. Reply ONLY with a JSON object -- no prose. "
            f"Table: {tbl}. Infer the most likely source timezone for datetime values. "
            "Context: Athena/EHR clinical data -> Eastern; "
            "phone-system, Lightbeam, web/API -> UTC; financial/operational -> Eastern or UTC. "
            f"Sample columns: {payload}. "
            '{"inferred_tz":"Eastern|UTC|unknown","confidence":"high|medium|low","rationale":"<1 sentence>"}'
        )
        try:
            result = json.loads(_ai_query(prompt))
            inferred = result.get("inferred_tz", "unknown")
            tz_conf  = result.get("confidence", "low")
            for col in no_tz:
                if col in date_candidates[tbl]:
                    date_candidates[tbl][col]["inferred_source_tz"]    = inferred
                    date_candidates[tbl][col]["inferred_tz_confidence"] = tz_conf
        except Exception as e:
            print(f"  [WARN] AI TZ inference failed for {tbl}: {e}")


# ── Run classifier ────────────────────────────────────────────────────────────
for tbl in list(date_candidates.keys()):
    classify_table(tbl)
    remaining = len(date_candidates[tbl])
    print(f"  ok {tbl}: {remaining} confirmed date/datetime candidate(s)")

total_confirmed = sum(len(v) for v in date_candidates.values())
print(f"\nAI classification done. {total_confirmed} confirmed date column(s).")


In [0]:
# ---------------------------------------------------------------------------
# SCALABLE RULE ENGINE:
# All aggregate stats (sentinel counts, future dates, plausibility) come from
# the pre-computed dict in date_candidates -- no additional full-table SQL.
# Only the DOD<DOB cross-column check runs a query (on sample DF first).
# Magic date check runs on the cached sample DF.
# ---------------------------------------------------------------------------
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

def _esc(name: str) -> str:
    return name.replace("`", "``")

# Date-only format sets (shared by _check_format and _check_timezone)
DATE_ONLY_FMTS = frozenset([
    "YYYY-MM-DD", "MM/DD/YYYY", "DD/MM/YYYY", "DD-MM-YYYY",
    "YYYYMMDD", "typed_date", "DD.MM.YYYY", "YYYY/MM/DD"
])


def _finding(tbl, col, info, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = (auto_decided_action
               if auto_decided_action is not None and len(options) == 1
               else None)
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "data_type":                info.get("dtype", "UNKNOWN"),
        "layer":                    CFG["layer"],
        "detected_format":          info.get("detected_format", "unknown"),
        "semantic_type":            info.get("semantic_type", "other"),
        "timezone_context":         info.get("timezone_context", "unknown"),
        "has_est_companion":        bool(info.get("has_est_companion", False)),
        "has_source_tz_companion":  bool(info.get("has_source_tz_companion", False)),
        "semantic_source":          info.get("semantic_source", "heuristic"),
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


# ── §Format / §Validation: parse failures and format checks ──────────────────
def _check_format(tbl, col, info) -> list:
    dtype      = info["dtype"]
    total      = info["total"]
    parse_fail = info["parse_fail"]
    null_count = info["null_count"]
    non_null   = total - null_count
    fmt        = info.get("detected_format", "unknown")
    findings   = []

    is_date_only_fmt = fmt in DATE_ONLY_FMTS

    # High parse failure rate
    if non_null > 0:
        fail_pct = parse_fail / non_null * 100
        if fail_pct > CFG["parse_failure_threshold"]:
            findings.append(_finding(tbl, col, info, "HIGH_PARSE_FAILURE_RATE",
                parse_fail, total, [],
                f"{fail_pct:.1f}% of non-null values fail to parse. "
                "Automated DQ checks must verify parse success rate (§Validation). "
                "Quarantine all parse failures; investigate source system.",
                confidence="high",
                auto_decided_action="quarantine_failures",
            ))

    # Invalid format / genuine parse failures.
    # PREVIOUSLY this was suppressed with "and not is_date_only_fmt", on the
    # assumption that a date-only-shaped format (e.g. YYYY-MM-DD, MM/DD/YYYY)
    # means any parse_fail rows are just a format mismatch already captured
    # by NON_ISO_DATE_FORMAT below. That assumption breaks precisely when
    # fmt IS an ISO format ("YYYY-MM-DD" / "YYYY-MM-DD_HH:MM:SS" / "typed_date"
    # / "typed_timestamp"): those formats are explicitly EXCLUDED from the
    # NON_ISO_* check below (correctly -- they're not non-ISO), so nothing
    # ever fired for genuinely unparseable values (e.g. "NOT_A_DATE") sitting
    # inside an otherwise-ISO string column. Combined with HIGH_PARSE_FAILURE_RATE
    # being threshold-gated (default 5%), a single bad value in a large table
    # could vanish from the report with NO finding at all -- directly against
    # §Format-dates: "Reject or quarantine values that do not parse to valid
    # calendar dates" (an unconditional requirement, not threshold-gated).
    # Fix: always raise INVALID_DATE_FORMAT/INVALID_DATETIME_FORMAT when
    # parse_fail > 0, regardless of format. This can coexist with a
    # NON_ISO_* finding on the same column -- they cover different actions
    # (reformat the column vs. quarantine the specific bad rows) and the
    # procedure lists them as separate requirements.
    if parse_fail > 0:
        tag = "INVALID_DATE_FORMAT" if dtype in DATE_DTYPES else "INVALID_DATETIME_FORMAT"
        if tag not in {f["classification"] for f in findings}:
            findings.append(_finding(tbl, col, info, tag,
                parse_fail, total, [],
                f"Values that do not parse as valid {'date' if dtype in DATE_DTYPES else 'datetime'}s detected. "
                "Quarantine non-parseable rows (§Format).",
                confidence="high",
                auto_decided_action="quarantine_invalid",
            ))

    # String column whose format the AI Classifier (Cell 5) never resolved --
    # either its ai_query() call errored (see ai_classification_error below)
    # or its response omitted this column. detected_format is still the
    # profiler's placeholder ("string_date_unknown"). Previously this fell
    # through to the NON_ISO_* branch below and got reported with a fake,
    # unusable "format" -- surface it honestly instead, so nobody applies a
    # Silver reformat transform against a format nobody actually confirmed.
    if _is_str(dtype) and fmt == "string_date_unknown":
        ai_err = info.get("ai_classification_error")
        findings.append(_finding(tbl, col, info, "FORMAT_CLASSIFICATION_INCOMPLETE",
            non_null, total, info.get("sample_vals", [])[:5],
            "AI format classification did not complete for this confirmed date-like "
            "STRING column; detected_format is unresolved."
            + (f" Classifier error: {ai_err}" if ai_err else "")
            + " Confirm the actual source format (see sample_values) before applying "
              "any Silver reformat transform, then re-run classification (§Format).",
            confidence="low",
        ))
        return findings

    # Non-ISO format for string columns: use detected_format for tag selection,
    # not semantic_type string matching (more reliable)
    if _is_str(dtype) and fmt not in ("typed_date","typed_timestamp","YYYY-MM-DD",
                                       "YYYY-MM-DD_HH:MM:SS","unknown","other_format"):
        tag = ("NON_ISO_DATE_FORMAT" if fmt in DATE_ONLY_FMTS
               else "NON_ISO_DATETIME_FORMAT")
        findings.append(_finding(tbl, col, info, tag,
            non_null, total, info.get("sample_vals", [])[:5],
            f"String column stores {'date' if tag == 'NON_ISO_DATE_FORMAT' else 'datetime'}s "
            f"in non-ISO format ({fmt}). Silver must use ISO 8601 (§Format). "
            "Parse from detected format and reformat in the Silver transform.",
        ))

    return findings


# ── §TZ-normalization / §Documentation ───────────────────────────────────────
def _check_timezone(tbl, col, info) -> list:
    dtype    = info["dtype"]
    total    = info["total"]
    tz_ctx   = info.get("timezone_context", "unknown")
    sem      = info.get("semantic_type", "other")
    findings = []

    # Date-only columns are timezone-agnostic
    is_date_only = (dtype in DATE_DTYPES or sem == "date_only"
                    or info.get("detected_format", "") in DATE_ONLY_FMTS)
    if is_date_only:
        return []

    has_est   = info.get("has_est_companion", False)
    has_utc   = info.get("has_source_tz_companion", False)
    source_tz = info.get("source_tz", info.get("inferred_source_tz", "unknown"))

    if tz_ctx != "_est" and not has_est:
        findings.append(_finding(tbl, col, info, "MISSING_EST_CANONICAL",
            total - info["null_count"], total, [],
            "Datetime column has no _est Eastern canonical column in this table. "
            "Every datetime model must have a _est canonical (§TZ-normalization). "
            "Add CONVERT_TIMEZONE('US/Eastern', col) AS <col>_est in Silver transform.",
            confidence="high",
            auto_decided_action="add_est_column",
        ))

    is_est_col  = (tz_ctx == "_est")
    est_present = is_est_col or has_est
    if est_present and not has_utc and source_tz in ("UTC","Pacific","Central","Mountain"):
        findings.append(_finding(tbl, col, info, "MISSING_SOURCE_TZ_COMPANION",
            total - info["null_count"], total, [],
            f"Source timezone inferred as {source_tz} but no source-tz companion (_utc) column found. "
            "Preserve original-timezone value in <col>_utc companion (§TZ-normalization).",
            confidence=info.get("inferred_tz_confidence", info.get("semantic_confidence","medium")),
        ))

    if tz_ctx == "no_suffix" and source_tz in ("unknown","local_unknown"):
        findings.append(_finding(tbl, col, info, "UNDOCUMENTED_TIMEZONE",
            total - info["null_count"], total, [],
            "Datetime column has no timezone suffix and source timezone cannot be determined. "
            "Data dictionary must list timezone and source system clock (§Documentation). "
            "Add appropriate suffix (_est or _utc) after confirming source timezone.",
            confidence=info.get("inferred_tz_confidence","low"),
        ))

    return findings


# ── §DOB / §DOD preservation override ────────────────────────────────────────
# Uses pre-computed stats from the profiler -- NO additional full table scans.
def _check_dob_dod(tbl, col, info, dob_col: Optional[str], sample_df: DataFrame) -> list:
    sem   = info.get("semantic_type", "other")
    total = info["total"]
    findings = []

    if sem not in ("date_of_birth", "date_of_death"):
        return []

    c_cobalt = info.get("c_cobalt", 0)
    c_epoch  = info.get("c_epoch", 0)
    c_max    = info.get("c_max", 0)
    c_mosaic = info.get("c_mosaic", 0)
    c_future = info.get("c_future", 0)
    c_old    = info.get("c_old", 0)
    c_young  = info.get("c_young", 0)

    if c_cobalt:
        findings.append(_finding(tbl, col, info, "KNOWN_NULL_SENTINEL_PRESENT",
            c_cobalt, total, ["1875-05-20"],
            "Registered Cobalt known-null sentinel (1875-05-20). "
            "This is the ONLY case where DOB/DOD may be auto-nulled (§DOB-DOD).",
            confidence="high",
            auto_decided_action="convert_to_null",
        ))

    if sem == "date_of_death" and (c_mosaic + c_max) > 0:
        findings.append(_finding(tbl, col, info, "DOD_SENTINEL_USED_FOR_ALIVE",
            c_mosaic + c_max, total,
            [s for s, n in [("2199-01-01",c_mosaic),("9999-12-31",c_max)] if n],
            "POLICY VIOLATION: future sentinel in DOD means 'alive' -- forbidden. "
            "Living = NULL DOD; never substitute a future sentinel (§DOD-preservation).",
            confidence="high",
            auto_decided_action="convert_to_null",
        ))

    suspect = c_epoch + (c_max if sem == "date_of_birth" else 0)
    if suspect:
        findings.append(_finding(tbl, col, info, "SUSPECT_SENTINEL_DATE",
            suspect, total,
            [s for s, n in [("1900-01-01",c_epoch),("9999-12-31",c_max)] if n and
             (sem == "date_of_birth" or s == "1900-01-01")],
            "Unregistered suspect sentinel in DOB/DOD. "
            "PRESERVATION OVERRIDE: quarantine, never auto-null (§DOB-DOD).",
            confidence="high",
            auto_decided_action="quarantine_pending_investigation",
        ))

    if c_future:
        tag = "DOB_FUTURE_DATE" if sem == "date_of_birth" else "DOD_FUTURE_DATE"
        findings.append(_finding(tbl, col, info, tag,
            c_future, total, [],
            "Future date in DOB/DOD -- clinically impossible. "
            "PRESERVATION OVERRIDE: quarantine, never silently null (§DOB-preservation).",
            confidence="high",
            auto_decided_action="quarantine",
        ))

    if sem == "date_of_birth" and (c_old + c_young) > 0:
        findings.append(_finding(tbl, col, info, "DOB_PLAUSIBILITY",
            c_old + c_young, total, [],
            f"DOB implies age < 0 or > {CFG['age_max']} years. "
            "PRESERVATION OVERRIDE: quarantine, never null (§DOB-preservation).",
            confidence="high",
            auto_decided_action="quarantine",
        ))

    # DOD < DOB cross-column check: sample first, promote to full count if violations found
    if sem == "date_of_death" and dob_col and dob_col in date_candidates.get(tbl, {}):
        sample_violations = dod_before_dob_sample_check(
            tbl, col, dob_col, info, date_candidates[tbl][dob_col], sample_df
        )
        if sample_violations > 0:
            # Promote to full count on the full table
            try:
                fq     = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
                dob_i  = date_candidates[tbl][dob_col]
                dob_dt = dob_i["dtype"]
                dc     = (f"`{_esc(col)}`" if info["dtype"] in (DATE_DTYPES | TS_DTYPES)
                          else f"TRY_CAST(`{_esc(col)}` AS DATE)")
                dob_dc = (f"`{_esc(dob_col)}`" if dob_dt in (DATE_DTYPES | TS_DTYPES)
                          else f"TRY_CAST(`{_esc(dob_col)}` AS DATE)")
                cross = spark.sql(f"""
                    SELECT COUNT(*) AS n FROM {fq}
                    WHERE TRY_CAST({dc} AS DATE) IS NOT NULL
                      AND TRY_CAST({dob_dc} AS DATE) IS NOT NULL
                      AND TRY_CAST({dc} AS DATE) < TRY_CAST({dob_dc} AS DATE)
                """).collect()[0]["n"]
                if cross:
                    findings.append(_finding(tbl, col, info, "DOB_DOD_CHRONOLOGY",
                        cross, total, [f"DOD < DOB (vs. {dob_col})"],
                        f"DOD < DOB in {cross:,} row(s). Chronologically impossible. "
                        "PRESERVATION OVERRIDE: quarantine both, never alter values (§DOB-DOD).",
                        confidence="high",
                        auto_decided_action="quarantine",
                    ))
            except Exception as e:
                print(f"  [WARN] DOD<DOB full count failed for {tbl}.{col}: {e}")

    return findings


# ── §Format: generic sentinel dates (non-DOB/DOD) -- uses pre-computed stats ──
def _check_generic_sentinels(tbl, col, info, sample_df: DataFrame) -> list:
    sem   = info.get("semantic_type", "other")
    total = info["total"]
    findings = []

    if sem in ("date_of_birth", "date_of_death"):
        return []

    c_max    = info.get("c_max", 0)
    c_mosaic = info.get("c_mosaic", 0)

    if c_max and sem == "open_ended_end_date":
        findings.append(_finding(tbl, col, info, "UNAPPROVED_OPEN_ENDED_SENTINEL",
            c_max, total, ["9999-12-31"],
            "9999-12-31 in open-ended end-date column should be 2199-01-01 (§Format). "
            "9999-12-31 overflows downstream tools and distorts date math.",
            confidence="high",
            auto_decided_action="replace_with_mosaic_sentinel",
        ))

    # Magic date: uses sample DF (no additional full scan)
    for dv, cnt in magic_date_check(tbl, col, info, sample_df):
        findings.append(_finding(tbl, col, info, "MAGIC_DATE_DETECTED",
            cnt, total, [str(dv)],
            f"Date {dv} appears in >= {CFG['magic_pct']}% of sampled rows. "
            "Possible system default or undocumented sentinel. "
            "Do not use magic dates without a dictionary entry (§Format-datetimes).",
            confidence="medium",
        ))

    return findings


# ── §Validation: future clinical event dates -- uses pre-computed c_future ────
def _check_clinical_future(tbl, col, info) -> list:
    sem     = info.get("semantic_type", "other")
    total   = info["total"]
    c_future = info.get("c_future", 0)

    if sem != "clinical_event_date" or c_future == 0:
        return []

    return [_finding(tbl, col, info, "FUTURE_CLINICAL_EVENT",
        c_future, total, [],
        f"Clinical event date has {c_future:,} future value(s). "
        "Recorded clinical events cannot occur in the future (§Validation). "
        "Quarantine future-dated rows; investigate source.",
        confidence="high",
    )]


# ── Table-level TZ consistency check ─────────────────────────────────────────
def _check_tz_consistency(tbl) -> list:
    ts_cols = [col for col, info in date_candidates.get(tbl,{}).items()
               if info["dtype"] in TS_DTYPES and info.get("semantic_type") != "date_only"]
    if len(ts_cols) < 2:
        return []
    tz_ctxs  = {date_candidates[tbl][col]["timezone_context"] for col in ts_cols}
    has_named = bool(tz_ctxs & {"_est","_utc","other_tz"})
    has_none  = "no_suffix" in tz_ctxs
    if not (has_named and has_none):
        return []
    unnamed = [c for c in ts_cols if date_candidates[tbl][c]["timezone_context"] == "no_suffix"]
    ref_info = date_candidates[tbl][ts_cols[0]]
    return [_finding(tbl, "-- multiple columns --", ref_info, "INCONSISTENT_TZ_SUFFIX",
        len(unnamed), len(ts_cols), unnamed[:5],
        f"Table has {len(ts_cols)} datetime columns with inconsistent TZ suffix conventions. "
        f"{len(unnamed)} column(s) lack a TZ suffix while others have one. "
        "Use _est / _utc naming consistently across all columns (§TZ-normalization).",
        confidence="high",
    )]


# ── Main loop -- no additional full-table SQL except DOD<DOB promotion ────────
all_findings: List[dict] = []

for tbl, cols in date_candidates.items():
    sample_df = table_samples.get(tbl)
    if sample_df is None:
        continue
    tbl_count = 0

    dob_col = next(
        (c for c, i in cols.items() if i.get("semantic_type") == "date_of_birth"),
        None
    )

    all_findings.extend(_check_tz_consistency(tbl))

    for col, info in cols.items():
        col_findings = (
            _check_format(tbl, col, info)
            + _check_timezone(tbl, col, info)
            + _check_dob_dod(tbl, col, info, dob_col, sample_df)
            + _check_generic_sentinels(tbl, col, info, sample_df)
            + _check_clinical_future(tbl, col, info)
        )
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)

    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("detected_format",          StringType(),  True),
    StructField("semantic_type",            StringType(),  True),
    StructField("timezone_context",         StringType(),  True),
    StructField("has_est_companion",        BooleanType(), True),
    StructField("has_source_tz_companion",  BooleanType(), True),
    StructField("semantic_source",          StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
# Section 1 -- Blocking findings
# Section 2 -- By classification
# Section 3 -- By table
# Section 4 -- Timezone gaps
# Section 5 -- DOB / DOD issues
# Section 6 -- Engineer work queue

BLOCKING = {
    "INVALID_DATE_FORMAT", "INVALID_DATETIME_FORMAT", "HIGH_PARSE_FAILURE_RATE",
    "DOB_FUTURE_DATE", "DOD_FUTURE_DATE", "DOB_DOD_CHRONOLOGY",
    "DOD_SENTINEL_USED_FOR_ALIVE", "SUSPECT_SENTINEL_DATE", "MISSING_EST_CANONICAL"
}

if not all_findings:
    print("No date/datetime columns found matching criteria. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 — BLOCKING FINDINGS: {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","data_type","semantic_type","detected_format",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 — Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("affected_rows"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 70)
    print("SECTION 3 — Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.countDistinct("column_name").alias("date_columns"),
               F.sum(F.when(~F.col("has_est_companion") & (F.col("data_type") != "DATE"), 1).otherwise(0)).alias("missing_est"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- Timezone gaps
    tz_df = fdf.filter(F.col("classification").isin(
        "MISSING_EST_CANONICAL","MISSING_SOURCE_TZ_COMPANION","INCONSISTENT_TZ_SUFFIX","UNDOCUMENTED_TIMEZONE"
    ))
    n_tz = tz_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 — Timezone gaps ({n_tz})")
    print("  Every datetime model must have a _est canonical; non-Eastern sources need _utc companion")
    print("-" * 70)
    if n_tz:
        display(tz_df.select(
            "table_name","column_name","data_type","timezone_context",
            "has_est_companion","has_source_tz_companion","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None.")

    # Section 5 -- DOB / DOD issues
    dob_dod_tags = {
        "DOB_FUTURE_DATE","DOD_FUTURE_DATE","DOB_DOD_CHRONOLOGY","DOB_PLAUSIBILITY",
        "SUSPECT_SENTINEL_DATE","KNOWN_NULL_SENTINEL_PRESENT","DOD_SENTINEL_USED_FOR_ALIVE"
    }
    dd_df = fdf.filter(F.col("classification").isin(dob_dod_tags))
    n_dd  = dd_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 — DOB / DOD preservation issues ({n_dd})")
    print("  PHI preservation override: quarantine suspect values, never silently null (except 1875-05-20)")
    print("-" * 70)
    if n_dd:
        display(dd_df.select(
            "table_name","column_name","semantic_type","classification",
            "affected_count","affected_pct","recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name"))
    else:
        print("  None.")

    # Section 6 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 — Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query results: {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Date columns scanned: {total_confirmed}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  TZ gaps: {n_tz}  |  DOB/DOD: {n_dd}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
